# UCS420 — Cognitive Computing | Assignment 10

**Name:** Aanya Garg  
**Roll Number:** 1024170172  
**Assistant:** Heart Health Information Assistant *(even roll number)*




In [ ]:

import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


USE_NLTK = False
try:
    import nltk
    from nltk.stem import WordNetLemmatizer
    try:
        nltk.data.find("corpora/wordnet")
    except LookupError:
        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
    lemmatizer = WordNetLemmatizer()
  
    lemmatizer.lemmatize("cars")
    USE_NLTK = True
except Exception as error:
    print("NLTK WordNet is unavailable; using the included lightweight fallback.")

print("Libraries imported successfully.")
print("NLTK WordNet lemmatization available:", USE_NLTK)

## 2. Create a labelled corpus



In [ ]:
data = {
    "general_information": [
        "What does heart health mean?",
        "Explain how the human heart works",
        "What is the function of the heart?",
        "Tell me about the cardiovascular system",
        "What is a healthy heart rate?",
        "Why is the heart important for the body?",
        "What is blood circulation?",
        "Explain the meaning of blood pressure",
        "What are the chambers of the heart?",
        "How does the heart pump blood?"
    ],
    "symptoms": [
        "What are common signs of heart disease?",
        "Can shortness of breath be related to the heart?",
        "Why do I feel chest discomfort?",
        "What symptoms can indicate an irregular heartbeat?",
        "Can unusual tiredness be a heart symptom?",
        "What does swelling in the ankles mean for heart health?",
        "Can dizziness be associated with heart problems?",
        "What are warning signs of poor circulation?",
        "How can palpitations feel?",
        "What symptoms should I discuss with a doctor?"
    ],
    "risk_factors": [
        "Does smoking increase heart disease risk?",
        "Can high blood pressure affect the heart?",
        "How does diabetes affect cardiovascular health?",
        "Is family history a risk factor for heart disease?",
        "Does high cholesterol increase heart risk?",
        "Can lack of exercise harm the heart?",
        "How does stress affect cardiovascular health?",
        "Does obesity increase the risk of heart disease?",
        "Can age influence heart disease risk?",
        "Is poor sleep linked with heart health?"
    ],
    "prevention": [
        "How can I reduce my risk of heart disease?",
        "What habits help prevent heart problems?",
        "How often should adults exercise for heart health?",
        "Can quitting smoking protect my heart?",
        "How can blood pressure be managed?",
        "What checkups help monitor heart health?",
        "How can I maintain a healthy weight?",
        "What lifestyle changes support the heart?",
        "How can I reduce cholesterol naturally?",
        "What are simple steps for a healthier heart?"
    ],
    "nutrition_lifestyle": [
        "Which foods are good for heart health?",
        "Is eating too much salt harmful to the heart?",
        "What is a heart healthy diet?",
        "Are fruits and vegetables good for the heart?",
        "Why should I limit saturated fat?",
        "How much water should I drink daily?",
        "Is walking beneficial for cardiovascular fitness?",
        "Does regular sleep support heart health?",
        "What are examples of healthy fats?",
        "Can whole grains be part of a heart healthy diet?"
    ],
    "monitoring_and_care": [
        "How is blood pressure measured?",
        "What is an ECG test used for?",
        "Why might a doctor request a cholesterol test?",
        "How can I keep a record of my blood pressure?",
        "What should I ask my doctor about heart health?",
        "What happens during a routine heart checkup?",
        "What does a pulse oximeter measure?",
        "Why is it important to follow medical advice?",
        "How often should I review my heart risk with a doctor?",
        "What information should I bring to a cardiology appointment?"
    ]
}

records = [
    {"query": query, "intent": intent}
    for intent, queries in data.items()
    for query in queries
]
corpus = pd.DataFrame(records)

print("Number of labelled queries:", len(corpus))
print("Number of intent categories:", corpus["intent"].nunique())
display(corpus.groupby("intent").size().rename("Number of queries").to_frame())
display(corpus.head(10))

## 3. Hierarchical intent taxonomy



In [ ]:
taxonomy = {
    "Heart basics": {
        "General information": ["general_information"],
        "Heart structure and function": ["general_information"]
    },
    "Symptoms and warning signs": {
        "Possible symptoms": ["symptoms"],
        "Urgent warning signs": ["symptoms"]
    },
    "Risk and prevention": {
        "Risk factors": ["risk_factors"],
        "Prevention strategies": ["prevention"]
    },
    "Healthy living and healthcare": {
        "Nutrition and physical activity": ["nutrition_lifestyle"],
        "Tests, monitoring and medical care": ["monitoring_and_care"]
    }
}

for major, subcategories in taxonomy.items():
    print(major)
    for sub, labels in subcategories.items():
        print("   └──", sub, "→", ", ".join(labels))

## 4. Domain concepts and relationships



In [ ]:
domain_concepts = [
    "Heart", "Blood", "Blood pressure", "Cholesterol", "Blood vessels",
    "Physical activity", "Nutrition", "Smoking", "Diabetes", "ECG",
    "Healthcare professional", "Cardiovascular risk"
]

relationships = [
    ("Heart", "pumps", "Blood"),
    ("Blood", "flows through", "Blood vessels"),
    ("Blood pressure", "describes pressure within", "Blood vessels"),
    ("Cholesterol", "can contribute to plaque in", "Blood vessels"),
    ("Smoking", "increases", "Cardiovascular risk"),
    ("Diabetes", "is associated with increased", "Cardiovascular risk"),
    ("Physical activity", "supports", "Heart health"),
    ("Nutrition", "influences", "Cholesterol"),
    ("ECG", "records electrical activity of", "Heart"),
    ("Healthcare professional", "interprets", "ECG"),
    ("Healthcare professional", "assesses", "Cardiovascular risk"),
    ("Blood pressure", "is monitored by", "Healthcare professional")
]

print("Number of concepts:", len(domain_concepts))
print("Number of relationships:", len(relationships))
display(pd.DataFrame(relationships, columns=["Concept 1", "Relationship", "Concept 2"]))

## 5. Text preprocessing



In [ ]:
def fallback_lemma(token):
    """A small rule-based fallback; not as comprehensive as WordNet."""
    irregular = {"children": "child", "teeth": "tooth", "feet": "foot",
                 "people": "person", "men": "man", "women": "woman"}
    if token in irregular:
        return irregular[token]
    if len(token) > 4 and token.endswith("ies"):
        return token[:-3] + "y"
    if len(token) > 4 and token.endswith("ing"):
        return token[:-3]
    if len(token) > 3 and token.endswith("ed"):
        return token[:-2]
    if len(token) > 3 and token.endswith("s") and not token.endswith("ss"):
        return token[:-1]
    return token

def preprocess_text(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-z]+", text)
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]
    if USE_NLTK:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    else:
        tokens = [fallback_lemma(word) for word in tokens]
    return " ".join(tokens)

examples = [
    "The HEART is pumping blood through the vessels.",
    "Patients are checking their blood pressures."
]
for sentence in examples:
    print("Original:   ", sentence)
    print("Processed:  ", preprocess_text(sentence))
    print()

## 6. TF-IDF feature extraction and intent classifier



In [ ]:
X = corpus["query"]
y = corpus["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        preprocessor=preprocess_text,
        tokenizer=str.split,
        token_pattern=None,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

model.fit(X_train, y_train)
print("Training examples:", len(X_train))
print("Testing examples:", len(X_test))
print("Model training completed.")

## 7. Model evaluation



In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"],
    "Score": [accuracy, precision, recall, f1]
})
display(metrics.style.format({"Score": "{:.3f}"}))

print("Classification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = sorted(corpus["intent"].unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion Matrix — Heart Health Intent Classifier")
plt.xlabel("Predicted intent")
plt.ylabel("Actual intent")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Domain-specific knowledge base



In [ ]:
knowledge_base = {
    "general_information": (
        "The heart is a muscular organ that pumps blood around the body. "
        "The cardiovascular system includes the heart and blood vessels. "
        "Heart rate and blood pressure are different measurements and can vary "
        "with activity, age and other factors."
    ),
    "symptoms": (
        "Possible heart-related symptoms can include chest discomfort, shortness "
        "of breath, palpitations, unusual fatigue, dizziness or swelling. These "
        "symptoms can have many causes and cannot be diagnosed by this assistant. "
        "Discuss new, recurring or concerning symptoms with a healthcare professional."
    ),
    "risk_factors": (
        "Factors associated with cardiovascular risk include smoking, high blood "
        "pressure, high cholesterol, diabetes, physical inactivity, excess weight, "
        "family history and increasing age. Individual risk depends on several "
        "factors and should be assessed by a qualified professional."
    ),
    "prevention": (
        "Heart-health measures include avoiding tobacco, being physically active "
        "within your ability, choosing a balanced eating pattern, getting adequate "
        "sleep, and having blood pressure, cholesterol and diabetes risk reviewed "
        "as appropriate. A clinician can help tailor a prevention plan."
    ),
    "nutrition_lifestyle": (
        "A heart-supportive eating pattern commonly emphasizes vegetables, fruits, "
        "whole grains, beans, nuts and appropriate sources of protein. Limiting "
        "excess salt, highly processed foods and saturated fats may be helpful. "
        "Choose activity and food changes that are safe and sustainable for you."
    ),
    "monitoring_and_care": (
        "A healthcare professional may use blood pressure measurement, cholesterol "
        "tests, an ECG and other assessments depending on the person's situation. "
        "Keep a record of readings if advised, bring a list of medicines and "
        "questions to appointments, and follow professional guidance."
    )
}

print("Knowledge base created for", len(knowledge_base), "intent categories.")

## 9. Reasoning rules and integrated assistant

The assistant applies the following rules:

1. **Emergency escalation:** emergency warning phrases are checked before classification.
2. **Unsupported-query handling:** non-heart-health topics are redirected.
3. **Low-confidence handling:** uncertain predictions are not presented as reliable intent matches.
4. **Knowledge retrieval:** a response is retrieved using the predicted intent.
5. **Medical-safety rule:** the assistant does not diagnose or prescribe.
6. **Professional-care rule:** personal or persistent symptoms are directed to a healthcare professional.

The emergency rule is deliberately checked first.

In [ ]:
EMERGENCY_KEYWORDS = [
    "severe chest pain", "crushing chest pain", "cannot breathe",
    "can't breathe", "difficulty breathing", "fainting", "unconscious",
    "stroke", "face drooping", "sudden weakness", "severe shortness of breath"
]

HEART_DOMAIN_KEYWORDS = {
    "heart", "cardiac", "cardiovascular", "blood pressure", "cholesterol",
    "pulse", "ecg", "ekg", "blood vessel", "circulation", "chest",
    "heartbeat", "palpitation", "cardiologist", "heart rate", "ticker", "walk", "diabetes",
    "smoking", "exercise", "nutrition", "diet", "symptom", "risk"
}

def is_emergency(query):
    q = query.lower()
    return any(term in q for term in EMERGENCY_KEYWORDS)

def is_supported_domain(query):
    q = query.lower()
    return any(term in q for term in HEART_DOMAIN_KEYWORDS)

def heart_health_assistant(query, confidence_threshold=0.30):
    query = str(query).strip()

    if not query:
        return "Please enter a question about heart health."

    if is_emergency(query):
        return (
            "EMERGENCY ALERT: This may describe a medical emergency. "
            "Call your local emergency number now (112 in India) or go to the "
            "nearest emergency department. Do not wait for an online response. "
            "This assistant cannot assess or treat emergencies."
        )

    if not is_supported_domain(query):
        return (
            "I can currently answer general questions about heart health, "
            "cardiovascular risk, prevention, nutrition and heart-related care. "
            "Please rephrase your question within these topics or consult an "
            "appropriate healthcare professional."
        )

    probabilities = model.predict_proba([query])[0]
    best_index = int(np.argmax(probabilities))
    predicted_intent = model.classes_[best_index]
    confidence = float(probabilities[best_index])

    if confidence < confidence_threshold:
        return (
            "I am not sufficiently confident about the meaning of that question. "
            "Please rephrase it with a little more detail. For personal medical "
            "advice, contact a qualified healthcare professional."
        )

    answer = knowledge_base.get(predicted_intent)
    if answer is None:
        return "I do not have a suitable information entry for that question."

    return (
        f"Detected intent: {predicted_intent} (confidence: {confidence:.2f})\n\n"
        f"{answer}\n\n"
        "Safety note: This is general educational information, not a diagnosis "
        "or a treatment recommendation. Consult a healthcare professional for "
        "personal advice."
    )

print("Integrated heart health assistant is ready.")

## 10. Test the assistant



In [ ]:
test_queries = [
    ("General information", "How does the heart pump blood?"),
    ("Symptoms", "I sometimes notice my heart beating irregularly; what could that mean?"),
    ("Risk factors", "Can tobacco use raise my cardiovascular risk?"),
    ("Prevention", "What daily habits can help protect my heart?"),
    ("Nutrition and lifestyle", "Suggest foods that are supportive of cardiovascular health."),
    ("Monitoring and care", "Why would a clinician perform an ECG?"),
    ("Differently worded query", "Does taking a brisk walk regularly benefit my ticker?"),
    ("Unsupported healthcare query", "How can I treat a skin rash?"),
    ("Low-confidence query", "heart thing maybe"),
    ("Emergency-related query", "I have severe chest pain and difficulty breathing.")
]

for label, query in test_queries:
    print("=" * 85)
    print("Test:", label)
    print("User:", query)
    if label == "Low-confidence query":
        print("Assistant:", heart_health_assistant(query, confidence_threshold=0.99))  # stricter threshold to demonstrate this safety branch
    else:
        print("Assistant:", heart_health_assistant(query))
    print()

## 11. Interactive interface — enter multiple queries



In [ ]:
print("HEART HEALTH INFORMATION ASSISTANT")
print("Ask a general heart-health question. Type 'exit' to stop.\n")

while True:
    user_query = input("You: ").strip()
    if user_query.lower() in {"exit", "quit", "stop"}:
        print("Assistant: Thank you for using the Heart Health Information Assistant.")
        break
    print("\nAssistant:", heart_health_assistant(user_query))
    print()